# Stock Price Trend Prediction: 0

## Downloading The Dataset and Extracting All Features From It.

<br>


### Part A: Import all libraries and downlaod the dataset.


In [8]:
import os
import shutil
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

In [9]:
# A1. Downloading and unziping

!kaggle datasets download -d matiflatif/apple-stock-complete-dataset
!unzip apple-stock-complete-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/matiflatif/apple-stock-complete-dataset
License(s): CC0-1.0
apple-stock-complete-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  apple-stock-complete-dataset.zip
  inflating: AAPL_1980-12-03_2025-02-07.csv  
  inflating: AAPL_1980-12-03_2025-02-18.csv  
  inflating: AAPL_1980-12-03_2025-02-24.csv  
  inflating: AAPL_1980-12-03_2025-03-02 (1).csv  
  inflating: AAPL_1980-12-03_2025-03-15.csv  
  inflating: AAPL_1980-12-13_2025-01-20.csv  
  inflating: AAPL_1980-12-13_2025-01-31.csv  


<br><br>

### Part B: Now calculating and appending all the necessary metrices


In [10]:
# B1. Checking which csv holds the most number of data rows.

files = [
    'AAPL_1980-12-03_2025-02-07.csv',
    'AAPL_1980-12-03_2025-03-15.csv',  # latest end date — use this one
]

for f in files:
    df = pd.read_csv(f)
    print(f, "→", df.shape)

AAPL_1980-12-03_2025-02-07.csv → (11129, 7)
AAPL_1980-12-03_2025-03-15.csv → (11154, 7)


In [11]:
# B2. Filtering and cleaning our fetched data.

data = pd.read_csv('AAPL_1980-12-03_2025-03-15.csv', index_col=0, parse_dates=True)

# Convert with utc=True first, then strip timezone
data.index = pd.to_datetime(data.index, utc=True).tz_convert(None)

# Filter from 2010
data = data[data.index >= '2010-01-01']

# Removes the time part
data.index = data.index.normalize()

data.head()

,open,high,low,close,adj_close,volume
date,,,,,,
2010-01-04,7.622500,7.660714,7.585000,7.643214,6.440329,493729600
2010-01-05,7.664286,7.699643,7.616071,7.656429,6.451466,601904800
2010-01-06,7.656429,7.686786,7.526786,7.534643,6.348847,552160000
2010-01-07,7.562500,7.571429,7.466071,7.520714,6.337111,477131200
2010-01-08,7.510714,7.571429,7.466429,7.570714,6.379242,447610800


In [12]:
# B3. Check the shape of data prior to appending metrices

print("Shape: ", data.shape)
print("\nColumns: ", data.columns)
print("\nTypes: ", data.dtypes)

Shape:  (3823, 6)

Columns:  Index(['open', 'high', 'low', 'close', 'adj_close', 'volume'], dtype='str')

Types:  open         float64
high         float64
low          float64
close        float64
adj_close    float64
volume         int64
dtype: object


In [13]:
# B4. Calculating all the necessary data using Adjusted Close market value

# ── Feature Engineering ────────────────────────────────────

# 1. Moving averages
data['ma5']  = data['adj_close'].rolling(5).mean()
data['ma20'] = data['adj_close'].rolling(20).mean()

# 2. MA crossover signal (when short crosses long)
data['ma_cross'] = (data['ma5'] > data['ma20']).astype(int)

# 3. Daily return %
data['return'] = data['adj_close'].pct_change()

# 4. Volume change %
data['volume_change'] = data['volume'].pct_change()

# 5. RSI-14
def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs       = avg_gain / avg_loss
    return    100 - (100 / (1 + rs))

data['rsi14'] = compute_rsi(data['adj_close'])

# 6. Bollinger Bands
data['bb_mid']   = data['adj_close'].rolling(20).mean()
data['bb_upper'] = data['bb_mid'] + 2 * data['adj_close'].rolling(20).std()
data['bb_lower'] = data['bb_mid'] - 2 * data['adj_close'].rolling(20).std()
data['bb_width'] = (data['bb_upper'] - data['bb_lower']) / data['bb_mid']  # volatility measure

# 7. MACD
ema12          = data['adj_close'].ewm(span=12).mean()
ema26          = data['adj_close'].ewm(span=26).mean()
data['macd']   = ema12 - ema26
data['signal'] = data['macd'].ewm(span=9).mean()

# 8. Target — 1 if tomorrow close > today close
data['target'] = (data['adj_close'].shift(-1) > data['adj_close']).astype(int)

# ── Drop NaN rows ──────────────────────────────────────────
data = data.dropna()

# ── Verify ─────────────────────────────────────────────────
print("Shape:", data.shape)
print("\nColumns:", data.columns.tolist())
print("\nSample:")
print(data[['adj_close', 'ma5', 'ma20', 'rsi14', 'macd', 'bb_width', 'target']].head(10))
print("\nTarget distribution:")
print(data['target'].value_counts(normalize=True).round(3))

Shape: (3804, 19)

Columns: ['open', 'high', 'low', 'close', 'adj_close', 'volume', 'ma5', 'ma20', 'ma_cross', 'return', 'volume_change', 'rsi14', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_width', 'macd', 'signal', 'target']

Sample:
            adj_close       ma5      ma20      rsi14      macd  bb_width  \
date                                                                       
2010-02-01   5.860126  6.018117  6.231361  38.180145 -0.074927  0.126264   
2010-02-02   5.894133  5.957449  6.204051  40.705386 -0.080788  0.131490   
2010-02-03   5.995550  5.905387  6.181255  41.111516 -0.077561  0.129604   
2010-02-04   5.779476  5.861812  6.152786  37.621138 -0.088819  0.139852   
2010-02-05   5.882093  5.882276  6.130036  42.533150 -0.089459  0.142663   
2010-02-08   5.841768  5.878604  6.103162  33.221040 -0.091708  0.143807   
2010-02-09   5.904065  5.880590  6.082217  37.283225 -0.088010  0.142919   
2010-02-10   5.871862  5.855853  6.063258  38.933567 -0.086382  0.144048   
2010-02-11

In [14]:
# B5. Check the new shape of data after appending metrices

print("Shape: ", data.shape)
print("\nColumns: ", data.columns)
print("\nTypes: ", data.dtypes)

Shape:  (3804, 19)

Columns:  Index(['open', 'high', 'low', 'close', 'adj_close', 'volume', 'ma5', 'ma20',
       'ma_cross', 'return', 'volume_change', 'rsi14', 'bb_mid', 'bb_upper',
       'bb_lower', 'bb_width', 'macd', 'signal', 'target'],
      dtype='str')

Types:  open             float64
high             float64
low              float64
close            float64
adj_close        float64
volume             int64
ma5              float64
ma20             float64
ma_cross           int64
return           float64
volume_change    float64
rsi14            float64
bb_mid           float64
bb_upper         float64
bb_lower         float64
bb_width         float64
macd             float64
signal           float64
target             int64
dtype: object


<br><br>

### Part C: Saving data back in google drive


In [16]:

file_path = os.path.join('Final_Extracted_Features.csv')
data.to_csv(file_path)

The `data` DataFrame has been saved successfully!
